# Tracking and simulating Iceberg Juliette's fate

#### Initialise notebook environement

In [1]:
from src.utils import *

# Configure the logging level to INFO
logging.basicConfig(level=logging.INFO)

In [2]:
HTML("""<div style="display: flex; justify-content: center; transform: scale(0.8);width: 100%; margin: auto;">
    <iframe src="https://nersc.no/en/features/juliette-helps-us-produce-better-forecasts/" width="100%" height="800px"></iframe>
</div>""")

HTML(value='<div style="display: flex; justify-content: center; transform: scale(0.8);width: 100%; margin: aut…

## Set study date
you may fix date here if need, eg. yyyy,mm,dd=(2025,5,27) 


In [3]:
yyyy,mm,dd=2025,10,20

## Show Juliette's track & ice berg detections from Sentinel-1

#### Load Juliette's GPS Data into notebook environment

In [4]:
!curl https://minio.dive.edito.eu/project-acciberg/juliette.geojson > ./juliette.geojson

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 32821  100 32821    0     0   711k      0 --:--:-- --:--:-- --:--:--  728k


#### Extract track from study date (2025-10-20) -1D to +2D

In [5]:
gdf = geojson_to_gdf_between_dates("https://minio.dive.edito.eu/project-acciberg/juliette.geojson",
                                  f"{yyyy}-{mm}-{dd-1}", f"{yyyy}-{mm}-{dd+3}")

#### Display Juliette's track in EDITO viewier and overlays with S-1 ice berg concentration maps (from Copernicus Marine)
➡️ close the detailed informations tab for visibility and use the slider to show the study date (2025-10-20)

<img src="../../images/27_EDITO_DOF_demo_viewer_snapshot.png" style="margin-left: 50px; width: 300px;" />

➡️ then download the `juliette.geojson` file and drag-and-drop it from your computer to the embedded EDITO viewer


In [7]:
HTML("""<div style="display: flex; justify-content: center; transform: scale(1.0);width: 100%; margin: auto;">
    <iframe src="https://viewer.dive.edito.eu/map?catalog=https:%2F%2Fapi.dive.edito.eu%2Fdata%2Fcatalogs%2Fcopernicus-marine-products%2Fcopernicus-marine-product-SEAICE_ARC_SEAICE_L4_NRT_OBSERVATIONS_011_007%2Fcopernicus-marine-dataset-DMI-ARC-SEAICE_BERG_MOSAIC_IW-L4-NRT-OBS&selected=https:%2F%2Fapi.dive.edito.eu%2Fdata%2Fcollections%2Fclimate_forecast-number_of_icebergs_per_unit_area%2Fitems%2F6d4fe831-f733-5c5b-9a09-e7341f7fa634&c=-21.1924037,59.950467,2.31" width="100%" height="600px"></iframe>
</div>""")

HTML(value='<div style="display: flex; justify-content: center; transform: scale(1.0);width: 100%; margin: aut…

## Run OpenBerg simulations

#### Initialize OpenBerg model

In [8]:
from opendrift.models.openberg import OpenBerg

o = OpenBerg(loglevel=20) 
o.set_config('drift:vertical_profile', False) # use surface currents for this test
o.set_config('drift:horizontal_diffusivity', 50)

09:35:45 INFO    opendrift:509: OpenDriftSimulation initialised (version 1.14.2)


#### Adding readers to forcing data (currents, waves, winds)
* using Arctic MFC TOPAZ5 hourly Arctic forecasts for currents
* using Arctic MFC 3km WAM model for waves
* using NCEP Global Atmospheric Model (GFS) forecasts for winds

In [9]:
o.add_readers_from_list([
    'cmems_mod_arc_phy_anfc_6km_detided_PT1H-i',
    'dataset-wam-arctic-1hr3km-be',
    'https://pae-paha.pacioos.hawaii.edu/thredds/dodsC/ncep_global/NCEP_Global_Atmospheric_Model_best.ncd'])

#### Seeding virtual particles on iceberg positions
* seeding 100 virtual particles, representing different iceberg characteristics
* Juliette is expected to range between 10 & 30m above sea level (and 50 to 300m below), with lengths and widths ranging between 20 to 130m approximately. Use a random function to represent random variability of these parameters.

In [10]:
_, start_idx=gdf_closest_date(gdf,f"{yyyy}-{mm}-{dd}",return_idx=True)
filt_gdf = gdf.iloc[[start_idx]] # we initialise on date closest to study date

In [11]:
n=100
randspace=np.random.rand(n)
lengths, widths, sails, drafts = randspace*80+50, randspace*50+20, randspace*20+10, randspace*250+50

icebergs = {'lon': filt_gdf.geometry.x, 'lat': filt_gdf.geometry.y,
            'sail': sails, 'draft': drafts, 'length': lengths, 'width': widths,
            'number': lengths.size, 'radius': 400,
           'time': filt_gdf.date.iloc[0].to_pydatetime()}

o.seed_elements(**icebergs)

09:35:48 INFO    opendrift.models.basemodel.environment:206: Adding a global landmask from GSHHG
09:35:51 INFO    opendrift.models.basemodel.environment:229: Fallback values will be used for the following variables which have no readers: 
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_floor_depth_below_sea_level: 10000.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_x_slope: 0.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_y_slope: 0.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_wave_significant_height: 0.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_wave_from_direction: 0.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_wave_stokes_drift_x_velocity: 0.000000
09:35:51 INFO    opendrift.models.basemodel.environment:232: 	sea_surface_wave_stokes_drift_y_velocity: 0.000000
09:35:51 INFO    opendrift.mode

#### Run simulations (over 2 days)

In [12]:
simulation_days = 2 #Modify here for extending/shortening simulation
ds=o.run(duration=timedelta(days=simulation_days))

09:36:42 INFO    opendrift:907: Using existing reader for land_binary_mask
09:36:42 INFO    opendrift:936: All points are in ocean
09:36:42 INFO    opendrift:2102: 2025-10-20 00:00:30 - step 1 of 48 - 100 active elements (0 deactivated)
09:36:42 INFO    opendrift.readers.reader_copernicusmarine:39: Using CMEMS password for user rdussurget_ext from environment variables COPERNICUSMARINE_SERVICE_USERNAME and COPERNICUSMARINE_SERVICE_PASSWORD
INFO - 2025-11-28T09:36:42Z - Selected dataset version: "202311"
09:36:42 INFO    copernicusmarine:251: Selected dataset version: "202311"
INFO - 2025-11-28T09:36:42Z - Selected dataset part: "default"
09:36:42 INFO    copernicusmarine:271: Selected dataset part: "default"
09:36:44 INFO    opendrift.readers.reader_netCDF_CF_generic:299: Grid coordinates are detected, but proj4 string not given: assuming latlong
09:36:44 INFO    opendrift.readers.reader_netCDF_CF_generic:332: Detected dimensions: {'y': 'latitude', 'x': 'longitude', 'time': 'time'}
09:

## Show results

In [13]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Your code here
import logging
logging.captureWarnings(False)
logger = logging.getLogger()
logger.setLevel(logging.ERROR)  # Only show ERROR level and above

#### Display virtual icebergs animation

In [14]:
o.animation(fast=True,filename='./simulation.gif',background=['x_sea_water_velocity', 'y_sea_water_velocity'],show_trajectories=True)
Image(url='./simulation.gif')

09:37:33 WARNING opendrift:2479: Plotting fast. This will make your plots less accurate.
09:37:35 INFO    opendrift:4650: Saving animation to ./simulation.gif...
09:38:08 INFO    opendrift:3086: Time to make animation: 0:00:34.173337


#### Dump simulated trajectories as geojson

In [15]:
write_geojson(ds,"./trajectories_topaz5.geojson")

#### Compare simulated tracks against observations
For this go back to EDITO viewer and add simulated tracks